# step 0 导包+定义路径+转换为parquet格式+核心字段定义

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import duckdb

In [3]:
#定义路径，导出到py文件里的时候要改成CWS = Path(__file__)
CWD = Path.cwd()
print(CWD)
import_path = CWD/"data"/"raw"/"accepted_2007_to_2018Q4.csv"
output_path = CWD/"outcomes"
output_path.mkdir(exist_ok=True, parents=True)

d:\Project


In [4]:
#字段按照发生时间分类(43个字段)
field_groups = {
    "loan_key": [
        "id",
        "member_id",
    ],

    "origination_terms": [
        "loan_amnt",
        "funded_amnt",
        "funded_amnt_inv",
        "term",
        "issue_d",
    ],

    "pricing": [
        "int_rate",
        "installment",
        "grade",
        "sub_grade",
    ],

    "borrower_profile": [
        "annual_inc",
        "emp_length",
        "home_ownership",
        "verification_status",
        "purpose",
        "addr_state",
        "dti",
    ],

    "credit_history_at_origination": [
        "delinq_2yrs",
        "earliest_cr_line",
        "fico_range_low",
        "fico_range_high",
        "inq_last_6mths",
        "open_acc",
        "pub_rec",
        "revol_bal",
        "revol_util",
        "total_acc",
        "pub_rec_bankruptcies",
    ],

    "loan_outcomes": [
        "loan_status",
        "out_prncp",
        "total_pymnt",
        "total_rec_prncp",
        "total_rec_int",
        "total_rec_late_fee",
        "recoveries",
        "collection_recovery_fee",
    ],

    "post_origination_updates": [
        "last_pymnt_d",
        "last_pymnt_amnt",
        "next_pymnt_d",
        "last_credit_pull_d",
        "last_fico_range_high",
        "last_fico_range_low",
    ],
}

#把按业务时间发生的字段提取到一个总的list
core_columns = []
for value in field_groups.values():
    core_columns.extend(value)

In [5]:
#分开定义核心的categorical字段与核心的numerical字段(29个字段)
categorical_cord_cols = [
    "term",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "verification_status",
    "purpose",
    "loan_status",
]

numerical_core_cols = [
    "loan_amnt",
    "funded_amnt",
    "funded_amnt_inv",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "fico_range_low",
    "fico_range_high",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "total_pymnt",
    "total_rec_prncp",
    "total_rec_int",
    "recoveries",
    "collection_recovery_fee",
]

# step1数据清洗

## step 1.1 基础 profiling 与 重复、丢失等信息

In [6]:
#把csc转换成parquet
df = pd.read_csv(import_path, low_memory=False)
df.to_parquet(import_path.parent/"data.parquet", index=False)
df = pd.read_parquet(import_path.parent/"data.parquet")

In [7]:
#查看43个按数据发生时间分类的字段是否都在数据里
missing_columns = []
for columns in field_groups.values():
    for column in columns:
        if column not in df.columns:
            missing_columns.append(column)
assert len(missing_columns) == 0, f"Missing columns: {missing_columns}"


In [8]:
#基础profiling
display(df.shape)
df.info()
display(df.describe(include='all'))
display(df.dtypes)

(2260701, 151)

<class 'pandas.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: float64(113), str(38)
memory usage: 3.1 GB


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
count,2260701,0.0,2.260668e+06,2.260668e+06,2.260668e+06,2260668,2.260668e+06,2.260668e+06,2260668,2260668,...,10917.000000,10917.000000,2260668,2260668,34246,34246,34246,34246.000000,34246.000000,34246.000000
unique,2260701,NaN,NaN,NaN,NaN,2,NaN,NaN,7,35,...,NaN,NaN,2,2,83,3,90,NaN,NaN,NaN
top,68407277,NaN,NaN,NaN,NaN,36 months,NaN,NaN,B,C1,...,NaN,NaN,Cash,N,Feb-2019,ACTIVE,Jan-2019,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,1609754,NaN,NaN,663557,145903,...,NaN,NaN,2182546,2226422,2606,14704,1710,NaN,NaN,NaN
mean,NaN,NaN,1.504693e+04,1.504166e+04,1.502344e+04,NaN,1.309283e+01,4.458068e+02,NaN,NaN,...,11636.883942,193.994321,NaN,NaN,NaN,NaN,NaN,5010.664267,47.780365,13.191322
std,NaN,NaN,9.190245e+03,9.188413e+03,9.192332e+03,NaN,4.832138e+00,2.671735e+02,NaN,NaN,...,7625.988281,198.629496,NaN,NaN,NaN,NaN,NaN,3693.122590,7.311822,8.159980
min,NaN,NaN,5.000000e+02,5.000000e+02,0.000000e+00,NaN,5.310000e+00,4.930000e+00,NaN,NaN,...,55.730000,0.010000,NaN,NaN,NaN,NaN,NaN,44.210000,0.200000,0.000000
25%,NaN,NaN,8.000000e+03,8.000000e+03,8.000000e+03,NaN,9.490000e+00,2.516500e+02,NaN,NaN,...,5627.000000,44.440000,NaN,NaN,NaN,NaN,NaN,2208.000000,45.000000,6.000000
50%,NaN,NaN,1.290000e+04,1.287500e+04,1.280000e+04,NaN,1.262000e+01,3.779900e+02,NaN,NaN,...,10028.390000,133.160000,NaN,NaN,NaN,NaN,NaN,4146.110000,45.000000,14.000000
75%,NaN,NaN,2.000000e+04,2.000000e+04,2.000000e+04,NaN,1.599000e+01,5.933200e+02,NaN,NaN,...,16151.890000,284.190000,NaN,NaN,NaN,NaN,NaN,6850.172500,50.000000,18.000000


id                           str
member_id                float64
loan_amnt                float64
funded_amnt              float64
funded_amnt_inv          float64
                          ...   
settlement_status            str
settlement_date              str
settlement_amount        float64
settlement_percentage    float64
settlement_term          float64
Length: 151, dtype: object

In [9]:
#核心重要字段的前五行内容展示和value_counts展示
format_check_cols = [
    "term",
    "issue_d",
    "int_rate",
    "emp_length",
    "loan_status",
]

for col in format_check_cols:
    print(f"\n===== {col} =====")
    print("dtype:", df[col].dtype)
    print("examples:")
    display(df[col].dropna().head(5))
    print("common values:")
    display(df[col].value_counts(dropna=False).head(10))


===== term =====
dtype: str
examples:


0     36 months
1     36 months
2     60 months
3     60 months
4     60 months
Name: term, dtype: str

common values:


term
 36 months    1609754
 60 months     650914
NaN                33
Name: count, dtype: int64


===== issue_d =====
dtype: str
examples:


...

emp_length
10+ years    748005
2 years      203677
< 1 year     189988
3 years      180753
1 year       148403
NaN          146940
5 years      139698
4 years      136605
6 years      102628
7 years       92695
Name: count, dtype: int64


===== loan_status =====
dtype: str
examples:


0    Fully Paid
1    Fully Paid
2    Fully Paid
3       Current
4    Fully Paid
Name: loan_status, dtype: str

common values:


loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64

In [10]:
#重复键统计
display(df.duplicated().sum())

np.int64(0)

In [11]:
#全体core_columns的缺失值、unique值统计
field_profile = pd.DataFrame({
    "dtype": df[core_columns].dtypes.astype(str),
    "missing_count": df[core_columns].isna().sum(),
    "missing_rate": df[core_columns].isna().mean().round(4),
    "unique_count": df[core_columns].nunique(dropna=False),
    "example_values": [df[col].dropna().head(5).tolist() for col in core_columns],
}).sort_values(by="missing_count", ascending=False)

display(field_profile)

,dtype,missing_count,missing_rate,unique_count,example_values
member_id,float64,2260701,1.0000,1,[]
next_pymnt_d,str,1345343,0.5951,107,"[Apr-2019, Apr-2019, Apr-2019, Apr-2019, Apr-2..."
emp_length,str,146940,0.0650,12,"[10+ years, 10+ years, 10+ years, 10+ years, 3..."
last_pymnt_d,str,2460,0.0011,137,"[Jan-2019, Jun-2016, Jun-2017, Feb-2019, Jul-2..."
revol_util,float64,1835,0.0008,1431,"[29.7, 19.2, 56.2, 11.6, 64.5]"
dti,float64,1744,0.0008,10846,"[5.91, 16.06, 10.78, 17.06, 25.37]"
pub_rec_bankruptcies,float64,1398,0.0006,13,"[0.0, 0.0, 0.0, 0.0, 0.0]"
last_credit_pull_d,str,105,0.0000,142,"[Mar-2019, Mar-2019, Mar-2019, Mar-2019, Mar-2..."
inq_last_6mths,float64,63,0.0000,29,"[1.0, 4.0, 0.0, 0.0, 3.0]"
open_acc,float64,62,0.0000,92,"[7.0, 22.0, 6.0, 13.0, 12.0]"


In [12]:
#numerical字段的初步EDA
numerical_profile = df[numerical_core_cols].describe(percentiles=[0.99,0.75,0.5,0.25,0.01]).round(3)
display(numerical_profile)

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,fico_range_low,fico_range_high,delinq_2yrs,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,total_pymnt,total_rec_prncp,total_rec_int,recoveries,collection_recovery_fee
count,2260668.000,2260668.000,2260668.000,2260668.000,2260668.000,2.260664e+06,2258957.000,2260668.000,2260668.000,2260639.000,...,2260639.000,2260639.000,2260668.000,2258866.000,2260639.000,2260668.000,2260668.000,2260668.000,2260668.000,2260668.000
mean,15046.931,15041.664,15023.438,13.093,445.807,7.799243e+04,18.824,698.588,702.588,0.307,...,11.612,0.198,16658.458,50.338,24.163,12082.557,9505.772,2431.388,143.879,23.983
std,9190.245,9188.413,9192.332,4.832,267.174,1.126962e+05,14.183,33.010,33.011,0.867,...,5.641,0.571,22948.305,24.713,11.988,9901.383,8321.852,2679.738,748.164,131.226
min,500.000,500.000,0.000,5.310,4.930,0.000000e+00,-1.000,610.000,614.000,0.000,...,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000
1%,1525.000,1516.750,1500.000,5.320,53.460,1.680000e+04,1.720,660.000,664.000,0.000,...,3.000,0.000,126.000,0.900,5.000,586.180,330.070,47.660,0.000,0.000
25%,8000.000,8000.000,8000.000,9.490,251.650,4.600000e+04,11.890,675.000,679.000,0.000,...,8.000,0.000,5950.000,31.500,15.000,4546.458,3000.000,728.188,0.000,0.000
50%,12900.000,12875.000,12800.000,12.620,377.990,6.500000e+04,17.840,690.000,694.000,0.000,...,11.000,0.000,11324.000,50.300,22.000,9329.720,7000.000,1525.940,0.000,0.000
75%,20000.000,20000.000,20000.000,15.990,593.320,9.300000e+04,24.490,715.000,719.000,0.000,...,14.000,0.000,20246.000,69.400,31.000,16940.869,13899.100,3108.062,0.000,0.000
99%,40000.000,40000.000,40000.000,26.770,1238.460,2.700000e+05,42.710,805.000,809.000,4.000,...,30.000,2.000,97929.980,98.100,60.000,42836.235,35000.000,13090.850,3453.880,598.500
max,40000.000,40000.000,40000.000,30.990,1719.830,1.100000e+08,999.000,845.000,850.000,58.000,...,101.000,86.000,2904836.000,892.300,176.000,63296.878,40000.000,28192.500,39859.550,7174.719


1.loan_amnt、funded_amount最低500，最高40000
2.annual_inc的平均数77992高于中位数65000，max=110000000明显不合理，需要探查
3.dti最低值小于0不合理，需要探查，最大值999不合理，需要探查
4.大部分人(70%)的delinq_2yrs为0，推测是因为大部分不为0的申请者被reject了；同理，大部分人的inq_last_6mths在1次以下，推测大量查询的用户的实际信用有限
5.大部分人的pub_rec为0
6.revol_util是比例，max超过100可能不合理，需要探查
7.大部分的recoveries为0，一个原因是大部分贷款是没有违约的


In [13]:
#categorical字段的EDA
categorical_results = []

for col in categorical_cord_cols:
    counts = df[col].astype(str).value_counts(dropna = False).rename_axis("value").reset_index(name="count")
    proportion = (100.0 * counts["count"]/len(df)).round(2)
    counts["proportion"] = proportion
    counts.insert(0,"yield",col)
    categorical_results.append(counts)
categorical_profile = pd.concat(categorical_results)
display(categorical_profile)


,yield,value,count,proportion
0,term,36 months,1609754,71.21
1,term,60 months,650914,28.79
2,term,NaN,33,0.00
0,grade,B,663557,29.35
1,grade,C,650053,28.75
...,...,...,...,...
5,loan_status,Late (16-30 days),4349,0.19
6,loan_status,Does not meet the credit policy. Status:Fully ...,1988,0.09
7,loan_status,Does not meet the credit policy. Status:Charge...,761,0.03
8,loan_status,Default,40,0.00


这几个字段的category都是合理的，但是同时存在33条missing value，33这个数字应该不是巧合，把33条记录拉出来看一下

In [14]:
#33条categorical字段的missing行
df_categorical_missing = df[df["loan_status"].isna()]
df_categorical_missing

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
421095,Total amount funded in policy code 1: 6417608175,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
421096,Total amount funded in policy code 2: 1944088810,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
528961,Total amount funded in policy code 1: 1741781700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
528962,Total amount funded in policy code 2: 564202131,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
651664,Total amount funded in policy code 1: 1791201400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
651665,Total amount funded in policy code 2: 651669342,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
749520,Total amount funded in policy code 1: 1443412975,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
749521,Total amount funded in policy code 2: 511988838,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
877716,Total amount funded in policy code 1: 2063142975,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
877717,Total amount funded in policy code 2: 823319310,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


果然，这33条记录是无效的，他们的id是“Total amount funded in policy code 2: 521953170”。后续要被删掉

In [15]:
#检查主键id
primary_key_profile = pd.Series(
    {"row_count":len(df),
     "missing_id_count":df['id'].isna().sum(),
     "duplicated_id":df["id"].duplicated().sum(),
     "unique_id_count":df["id"].nunique()}
)

In [16]:
#保存profiling结果
field_profile.to_csv(output_path/"field_profile")
numerical_profile.to_csv(output_path/"numerical_profile")
categorical_profile.to_csv(output_path/"categorical_profile",index=False)
primary_key_profile.to_csv(output_path/"primary_key_profile")

## stpe1.2 数据清洗

#清洗规则见docs/"cleaning_rules.md"

In [17]:
#删掉无意义33行和空id
original_len = len(df)
df_clean = df.copy()
df_clean["id"] = pd.to_numeric(df_clean["id"], errors='coerce')
df_clean = df_clean.dropna(subset="id")
df_clean = df_clean.drop("member_id",axis=1)
df_clean["id"]=df_clean["id"].astype("int64")
assert len(df)-len(df_clean) ==33,"被删行数量不一致"
assert len(df.columns.tolist())-len(df_clean.columns.tolist()) ==1,"被删列数量不一致"

In [18]:
#标准化term
df_clean["term"] = df_clean["term"].astype("string").str.strip()
df_clean["term_months"] =df_clean["term"].str.extract(r"(\d+)")[0].astype("Int64")
invalid_term_mask = (
    ~df_clean["term_months"].isin([36,60])|
    df_clean["term_months"].isna()
)
if invalid_term_mask.sum() > 0:
    print(f"发现{invalid_term_mask.sum()}个无效的term_months值,请检查:")
    display(df_clean.loc[invalid_term_mask,:])
elif invalid_term_mask.sum() == 0:
    print("term_months列解析成功,没有发现无效的值")
assert invalid_term_mask.sum() == 0, "存在无效的term_months值"

#标准化日期字段
date_columns = [
    "issue_d",
    "earliest_cr_line",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d",
]

date_parse_results = []
for col in date_columns:
    print(f"\n______{col}_______")
    date_cols_copy = df_clean[col].copy()
    date_cols_copy = pd.to_datetime(date_cols_copy,format= "%b-%Y",errors="coerce")
    parse_failure_count = date_cols_copy.isna().sum() - df_clean[col].isna().sum()
    parse_failure_mask = date_cols_copy.isna() & df_clean[col].notna()
    if parse_failure_count > 0:
        print(f"在{col}列中发现{parse_failure_count}个无法解析的值,请检查:")
        display(df_clean.loc[parse_failure_mask,["id",col]])
    elif parse_failure_count == 0:
        print(f"{col}列解析成功,没有发现无法解析的值")
    date_parse_results.append({"column":col,
                              "missing_count_after_parse":date_cols_copy.isna().sum(),
                              "parse_failure_count":parse_failure_count
                              }
                              )
    df_clean[col] = date_cols_copy
    assert parse_failure_count == 0, f"存在非空无法解析的{col}值"
display(pd.DataFrame(date_parse_results))
assert df_clean["issue_d"].isna().sum() == 0, "存在无法解析的issue_d值"

#把issue_d转换为月份格式
df_clean["issue_month"] = (
    df_clean["issue_d"]
    .dt.to_period("M")
    .astype("string")
)
#看一下有没有问题
display(
    df_clean[
        ["id", "term", "term_months", "issue_d", "issue_month"]
    ].head()
)

term_months列解析成功,没有发现无效的值

______issue_d_______
issue_d列解析成功,没有发现无法解析的值

______earliest_cr_line_______
earliest_cr_line列解析成功,没有发现无法解析的值

______last_pymnt_d_______
last_pymnt_d列解析成功,没有发现无法解析的值

______next_pymnt_d_______
next_pymnt_d列解析成功,没有发现无法解析的值

______last_credit_pull_d_______
last_credit_pull_d列解析成功,没有发现无法解析的值


,column,missing_count_after_parse,parse_failure_count
0,issue_d,0,0
1,earliest_cr_line,29,0
2,last_pymnt_d,2427,0
3,next_pymnt_d,1345310,0
4,last_credit_pull_d,72,0


,id,term,term_months,issue_d,issue_month
0,68407277,36 months,36,2015-12-01,2015-12
1,68355089,36 months,36,2015-12-01,2015-12
2,68341763,60 months,60,2015-12-01,2015-12
3,66310712,60 months,60,2015-12-01,2015-12
4,68476807,60 months,60,2015-12-01,2015-12


In [19]:
#统一数值字段
numeric_columns = [
    # 放款时字段
    "loan_amnt",
    "funded_amnt",
    "funded_amnt_inv",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "fico_range_low",
    "fico_range_high",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "pub_rec_bankruptcies",

    # 放款后结果字段：保留用于历史分析，但未来不能进 PD
    "out_prncp",
    "total_pymnt",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_amnt",
    "last_fico_range_low",
    "last_fico_range_high",
]

numeric_parse_results = []
for col in numeric_columns:
    print(f"\n______{col}_______")
    numeric_cols_copy = df_clean[col].copy()
    numeric_cols_copy = pd.to_numeric(numeric_cols_copy, errors="coerce")
    parse_failure_count = numeric_cols_copy.isna().sum() - df_clean[col].isna().sum()
    parse_failure_mask = numeric_cols_copy.isna() & df_clean[col].notna()
    if parse_failure_count > 0:
        print(f"在{col}列中发现{parse_failure_count}个无法解析的值,请检查:")
        display(df_clean.loc[parse_failure_mask,["id",col]])
    elif parse_failure_count == 0:
        print(f"{col}列解析成功,没有发现无法解析的值")
    numeric_parse_results.append({"column":col,
                              "missing_count_after_parse":numeric_cols_copy.isna().sum(),
                              "parse_failure_count":parse_failure_count
                              }
                             )
    df_clean[col] = numeric_cols_copy
    assert parse_failure_count == 0, f"存在非空无法解析的{col}值"


______loan_amnt_______
loan_amnt列解析成功,没有发现无法解析的值

______funded_amnt_______
funded_amnt列解析成功,没有发现无法解析的值

______funded_amnt_inv_______
funded_amnt_inv列解析成功,没有发现无法解析的值

______int_rate_______
int_rate列解析成功,没有发现无法解析的值

______installment_______
installment列解析成功,没有发现无法解析的值

______annual_inc_______
annual_inc列解析成功,没有发现无法解析的值

______dti_______
dti列解析成功,没有发现无法解析的值

______fico_range_low_______
fico_range_low列解析成功,没有发现无法解析的值

______fico_range_high_______
fico_range_high列解析成功,没有发现无法解析的值

______delinq_2yrs_______
delinq_2yrs列解析成功,没有发现无法解析的值

______inq_last_6mths_______
inq_last_6mths列解析成功,没有发现无法解析的值

______open_acc_______
open_acc列解析成功,没有发现无法解析的值

______pub_rec_______
pub_rec列解析成功,没有发现无法解析的值

______revol_bal_______
revol_bal列解析成功,没有发现无法解析的值

______revol_util_______
revol_util列解析成功,没有发现无法解析的值

______total_acc_______
total_acc列解析成功,没有发现无法解析的值

______pub_rec_bankruptcies_______
pub_rec_bankruptcies列解析成功,没有发现无法解析的值

______out_prncp_______
out_prncp列解析成功,没有发现无法解析的值

______total_pymnt_______
total_pymnt列

In [20]:
#不带百分号的百分数转小数
df_clean["int_rate_decimal"] = df_clean["int_rate"] / 100
df_clean["dti_decimal"] = df_clean["dti"] / 100
df_clean["revol_util_decimal"] = df_clean["revol_util"] / 100

In [21]:
#标记部分numerical_cols的异常值（比如过低过高收入、百分比超过0-100等）

#收入过大或为负
annual_inc_p999 = df_clean["annual_inc"].quantile(0.999)
df_clean["annual_inc_nonpositive_flag"] = df_clean["annual_inc"].le(0)
df_clean["annual_inc_top_0_1pct_flag"] = df_clean["annual_inc"].gt(annual_inc_p999)

#dti超过[0, 100]
df_clean["dti_out_of_range_flag"] =(df_clean["dti"].lt(0) | df_clean["dti"].gt(100))

#信用卡使用率超过[0, 100]
df_clean["revol_util_out_of_range_flag"] = (df_clean["revol_util"].lt(0) | df_clean["revol_util"].gt(100))

quality_summary_results = pd.DataFrame(
    {
        "metric": [
            "annual_inc_nonpositive",
            "annual_inc_top_0_1pct",
            "dti_out_of_range",
            "revol_util_out_of_range",
        ],
        "count": [
            int(df_clean["annual_inc_nonpositive_flag"].sum()),
            int(df_clean["annual_inc_top_0_1pct_flag"].sum()),
            int(df_clean["dti_out_of_range_flag"].sum()),
            int(df_clean["revol_util_out_of_range_flag"].sum()),
        ],
    }
)

display(quality_summary_results)

,metric,count
0,annual_inc_nonpositive,1667
1,annual_inc_top_0_1pct,2165
2,dti_out_of_range,2563
3,revol_util_out_of_range,7343


In [22]:
#清洗categorical字段
#去空格
strip_only_columns = [
    "emp_length",
    "verification_status",
    "purpose",
    "loan_status",
]
for col in strip_only_columns:
    df_clean[col] = df_clean[col].astype("string").str.strip()

#统一大小写字段
uppercase_columns = [
    "grade",
    "sub_grade",
    "home_ownership",
    "addr_state",
]
for col in uppercase_columns:
    df_clean[col] = df_clean[col].astype("string").str.upper().str.strip()

#就业年限
df_clean["emp_length_years"] = df_clean["emp_length"].replace(
        {
            "10+ years": 10,
            "9 years": 9,
            "8 years": 8,
            "7 years": 7,
            "6 years": 6,
            "5 years": 5,
            "4 years": 4,
            "3 years": 3,
            "2 years": 2,
            "1 year": 1,
            "< 1 year": 0.5
        }
        ).astype("Float64")

df_clean["emp_length_years"].value_counts(dropna=False).sort_index()

emp_length_years
0.5     189988
1.0     148403
2.0     203677
3.0     180753
4.0     136605
5.0     139698
6.0     102628
7.0      92695
8.0      91914
9.0      79395
10.0    748005
<NA>    146907
Name: count, dtype: Int64

In [23]:
#无效的grade、sub_grade、addr_state
invalid_grade_mask = df_clean["grade"].notna() & ~df_clean["grade"].isin(list("ABCDEFG"))

invalid_sub_grade_mask = df_clean["sub_grade"].notna()& ~df_clean["sub_grade"].str.fullmatch(r"[A-G][1-5]")

invalid_state_mask = df_clean["addr_state"].notna() & ~df_clean["addr_state"].str.fullmatch(r"[A-Z]{2}")

category_validation_results = pd.Series(
    {
        "invalid_grade_count": int(invalid_grade_mask.sum()),
        "invalid_sub_grade_count": int(invalid_sub_grade_mask.sum()),
        "invalid_state_format_count": int(invalid_state_mask.sum()),
    }
)

display(category_validation_results)

assert category_validation_results["invalid_grade_count"] == 0
assert category_validation_results["invalid_sub_grade_count"] == 0
assert category_validation_results["invalid_state_format_count"] == 0

invalid_grade_count           0
invalid_sub_grade_count       0
invalid_state_format_count    0
dtype: int64

In [24]:
categorical_clean_cols = [
    "term",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "verification_status",
    "purpose",
    "addr_state",
    "loan_status",
]

categorical_clean_results = []

for col in categorical_clean_cols:
    counts = (
        df_clean[col]
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="count")
    )
    counts.insert(0, "column", col)
    counts["proportion"] = (counts["count"] / len(df_clean)).round(6)
    categorical_clean_results.append(counts)

categorical_clean_audit = pd.concat(categorical_clean_results ,ignore_index=True)

display(categorical_clean_audit)

,column,value,count,proportion
0,term,36 months,1609754,0.71207
1,term,60 months,650914,0.28793
2,grade,B,663557,0.293523
3,grade,C,650053,0.287549
4,grade,A,433027,0.191548
...,...,...,...,...
134,loan_status,In Grace Period,8436,0.003732
135,loan_status,Late (16-30 days),4349,0.001924
136,loan_status,Does not meet the credit policy. Status:Fully ...,1988,0.000879
137,loan_status,Does not meet the credit policy. Status:Charge...,761,0.000337


In [25]:
#跨字段验证
# 1. FICO 区间：低端不能高于高端
df_clean["fico_range_order_flag"] = (
    df_clean["fico_range_low"].notna()
    & df_clean["fico_range_high"].notna()
    & (
        df_clean["fico_range_low"]
        > df_clean["fico_range_high"]
    )
)

# 2. 放款后的 FICO 区间：低端不能高于高端
df_clean["last_fico_range_order_flag"] = (
    df_clean["last_fico_range_low"].notna()
    & df_clean["last_fico_range_high"].notna()
    & (
        df_clean["last_fico_range_low"]
        > df_clean["last_fico_range_high"]
    )
)

# 3. 最早信用记录不能晚于贷款发放日
df_clean["credit_history_after_issue_flag"] = (
    df_clean["earliest_cr_line"].notna()
    & df_clean["issue_d"].notna()
    & (
        df_clean["earliest_cr_line"]
        >= df_clean["issue_d"]
    )
)

# 4. sub_grade 的首字母应与 grade 一致，例如 B3 对应 B
df_clean["grade_sub_grade_mismatch_flag"] = (
    df_clean["grade"].notna()
    & df_clean["sub_grade"].notna()
    & (
        df_clean["grade"]
        > df_clean["sub_grade"].str[0]
    )
)

# 5. 放款金额关系
df_clean["funded_exceeds_loan_flag"] = (
    df_clean["funded_amnt"].notna()
    & df_clean["loan_amnt"].notna()
    & (
        df_clean["funded_amnt"]
        > df_clean["loan_amnt"]
    )
)

df_clean["investor_funding_exceeds_funded_flag"] = (
    df_clean["funded_amnt_inv"].notna()
    & df_clean["funded_amnt"].notna()
    & (
        df_clean["funded_amnt_inv"]
        > df_clean["funded_amnt"]
    )
)

cross_field_audit_results = pd.DataFrame(
    {
        "metric": [
            "fico_range_low_greater_than_high",
            "last_fico_range_low_greater_than_high",
            "earliest_credit_line_after_issue_date",
            "grade_sub_grade_mismatch",
            "funded_amount_exceeds_loan_amount",
            "investor_funding_exceeds_funded_amount",
        ],
        "count": [
            int(df_clean["fico_range_order_flag"].sum()),
            int(df_clean["last_fico_range_order_flag"].sum()),
            int(df_clean["credit_history_after_issue_flag"].sum()),
            int(df_clean["grade_sub_grade_mismatch_flag"].sum()),
            int(df_clean["funded_exceeds_loan_flag"].sum()),
            int(
                df_clean[
                    "investor_funding_exceeds_funded_flag"
                ].sum()
            ),
        ],
    }
)

display(cross_field_audit_results)
assert cross_field_audit_results["count"].sum() == 0, "存在跨字段验证失败的情况"


,metric,count
0,fico_range_low_greater_than_high,0
1,last_fico_range_low_greater_than_high,0
2,earliest_credit_line_after_issue_date,0
3,grade_sub_grade_mismatch,0
4,funded_amount_exceeds_loan_amount,0
5,investor_funding_exceeds_funded_amount,0


In [26]:
#汇总本步骤结果并正式验收df_clean
# 1. 行数、主键、时间轴等最终检查
final_validation = pd.DataFrame(
    {
        "metric": [
            "raw_row_count",
            "clean_row_count",
            "footer_rows_removed",
            "missing_id_count",
            "duplicate_id_count",
            "missing_issue_d_count",
            "invalid_term_count",
        ],
        "count": [
            original_len,
            len(df_clean),
            original_len - len(df_clean),
            int(df_clean["id"].isna().sum()),
            int(df_clean["id"].duplicated().sum()),
            int(df_clean["issue_d"].isna().sum()),
            int((~df_clean["term_months"].isin([36, 60])).sum()),
        ],
    }
)

display(final_validation)

assert len(df_clean) == original_len - 33, "被删行数量不一致"
assert df_clean["id"].isna().sum() == 0, "存在缺失的id值"
assert df_clean["id"].duplicated().sum() == 0, "存在重复的id值"
assert df_clean["issue_d"].isna().sum() == 0, "存在缺失的issue_d值"
assert df_clean["term_months"].isin([36, 60]).all(), "存在无效的term_months值"
# 2. 检查 PD 候选特征是否存在缺失或泄露
pd_feature_candidates = [
    "loan_amnt",
    "funded_amnt",
    "funded_amnt_inv",
    "term_months",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "annual_inc",
    "emp_length_years",
    "home_ownership",
    "verification_status",
    "purpose",
    "addr_state",
    "dti",
    "fico_range_low",
    "fico_range_high",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "pub_rec_bankruptcies"
]

forbidden_pd_columns = [
    "loan_status",
    "out_prncp",
    "total_pymnt",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "last_fico_range_low",
    "last_fico_range_high"
]

missing_pd_candidates = (
    set(pd_feature_candidates) - set(df_clean.columns)
)

if missing_pd_candidates:
    raise ValueError(
        f"Missing PD candidate columns: {sorted(missing_pd_candidates)}"
    )

leakage_overlap = (
    set(pd_feature_candidates)
    & set(forbidden_pd_columns)
)

assert len(leakage_overlap) == 0, (f"Leakage found: {sorted(leakage_overlap)}")

print("Leakage boundary check passed.")

# 3. 汇总之前各步输出的 audit
# 日期解析 audit：list -> 长表
date_audit_long = (
    pd.DataFrame(date_parse_results)
    .melt(
        id_vars="column",
        value_vars=[
            "missing_count_after_parse",
            "parse_failure_count",
        ],
        var_name="metric_type",
        value_name="count",
    )
)

date_audit_long["section"] = "date_parsing"
date_audit_long["metric"] = (
    date_audit_long["column"]
    + "__"
    + date_audit_long["metric_type"]
)

date_audit_long = date_audit_long[
    ["section", "metric", "count"]
]


# 数值解析 audit：list -> 长表
numeric_audit_long = (
    pd.DataFrame(numeric_parse_results)
    .melt(
        id_vars="column",
        value_vars=[
            "missing_count_after_parse",
            "parse_failure_count",
        ],
        var_name="metric_type",
        value_name="count",
    )
)

numeric_audit_long["section"] = "numeric_parsing"
numeric_audit_long["metric"] = (
    numeric_audit_long["column"]
    + "__"
    + numeric_audit_long["metric_type"]
)

numeric_audit_long = numeric_audit_long[
    ["section", "metric", "count"]
]


# 数值异常 flag 汇总
quality_audit = quality_summary_results.copy()
quality_audit["section"] = "quality_flags"
quality_audit = quality_audit[
    ["section", "metric", "count"]
]


# 分类字段格式检查：Series -> DataFrame
category_audit = (
    category_validation_results
    .rename_axis("metric")
    .reset_index(name="count")
)

category_audit["section"] = "category_validation"
category_audit = category_audit[
    ["section", "metric", "count"]
]


# 跨字段检查
cross_field_audit = cross_field_audit_results.copy()
cross_field_audit["section"] = "cross_field_validation"
cross_field_audit = cross_field_audit[
    ["section", "metric", "count"]
]


# 最终行数、主键、时间轴检查
final_validation_audit = final_validation.copy()
final_validation_audit["section"] = "final_validation"
final_validation_audit = final_validation_audit[
    ["section", "metric", "count"]
]


# 合并为最终清洗审计表
cleaning_audit = pd.concat(
    [
        date_audit_long,
        numeric_audit_long,
        quality_audit,
        category_audit,
        cross_field_audit,
        final_validation_audit,
    ],
    ignore_index=True,
)

cleaning_audit["count"] = cleaning_audit["count"].astype("int64")

cleaning_audit = (
    cleaning_audit
    .sort_values(["section", "metric"])
    .reset_index(drop=True)
)

display(cleaning_audit)

,metric,count
0,raw_row_count,2260701
1,clean_row_count,2260668
2,footer_rows_removed,33
3,missing_id_count,0
4,duplicate_id_count,0
5,missing_issue_d_count,0
6,invalid_term_count,0


Leakage boundary check passed.


,section,metric,count
0,category_validation,invalid_grade_count,0
1,category_validation,invalid_state_format_count,0
2,category_validation,invalid_sub_grade_count,0
3,cross_field_validation,earliest_credit_line_after_issue_date,0
4,cross_field_validation,fico_range_low_greater_than_high,0
...,...,...,...
79,numeric_parsing,total_rec_prncp__parse_failure_count,0
80,quality_flags,annual_inc_nonpositive,1667
81,quality_flags,annual_inc_top_0_1pct,2165
82,quality_flags,dti_out_of_range,2563


In [27]:
#保存清洗审计表
cleaning_audit.to_csv(
    output_path / "cleaning_audit.csv",
    index=False,
)

print(f"Saved: {output_path / 'cleaning_audit.csv'}")

Saved: d:\Project\outcomes\cleaning_audit.csv


In [28]:
df_clean[["id", "issue_d", "issue_month"]].head().T

,0,1,2,3,4
id,68407277,68355089,68341763,66310712,68476807
issue_d,2015-12-01 00:00:00,2015-12-01 00:00:00,2015-12-01 00:00:00,2015-12-01 00:00:00,2015-12-01 00:00:00
issue_month,2015-12,2015-12,2015-12,2015-12,2015-12


# step2切分月度数据

## 2.1monthly batch audit

In [29]:
# 1. 只读取月度审计需要的三个字段
parquet_path = Path(
    "data/processed/accepted_loans_normalized.parquet"
)

batch_keys = pd.read_parquet(
    parquet_path,
    columns=["id", "issue_d", "issue_month"],
)

display(batch_keys.head())

print("数据形状：", batch_keys.shape)
print("\n字段类型：")
print(batch_keys.dtypes)


# 2. 检查基础完整性
total_row_count = len(batch_keys)
unique_loan_count = batch_keys["id"].nunique()
duplicate_loan_count = batch_keys["id"].duplicated().sum()
missing_id_count = batch_keys["id"].isna().sum()
missing_issue_d_count = batch_keys["issue_d"].isna().sum()
missing_issue_month_count = batch_keys["issue_month"].isna().sum()

print("\n基础检查：")
print("总行数：", total_row_count)
print("唯一贷款数：", unique_loan_count)
print("重复贷款数：", duplicate_loan_count)
print("缺失 id：", missing_id_count)
print("缺失 issue_d：", missing_issue_d_count)
print("缺失 issue_month：", missing_issue_month_count)


# 3. 验证 issue_month 是否由 issue_d 正确转换
expected_issue_months = (
    batch_keys["issue_d"]
    .dt.to_period("M")
    .astype("string")
)

issue_month_mismatch = (
    batch_keys["issue_month"] != expected_issue_months
)

issue_month_mismatch_count = issue_month_mismatch.sum()

print(
    "\nissue_month 转换不一致数量：",
    issue_month_mismatch_count,
)


# 4. 按发放月份汇总
monthly_group_results = (
    batch_keys
    .groupby("issue_month")
    .agg(
        total_issued_number=("id", "size"),
        unique_loan_number=("id", "nunique"),
        earliest_issue_date=("issue_d", "min"),
        latest_issue_date=("issue_d", "max"),
    )
    .sort_index()
    .reset_index(names="month")
)

# 每个月多出来的重复记录数量
monthly_group_results["duplicate_loan_number"] = (
    monthly_group_results["total_issued_number"]
    - monthly_group_results["unique_loan_number"]
)

# 截止每个月累计发放了多少笔贷款
monthly_group_results["cumulative_loan_count"] = (
    monthly_group_results["total_issued_number"].cumsum()
)

display(monthly_group_results.head())
display(monthly_group_results.tail())


# 5. 创建理论上应该存在的连续月份
expected_months = pd.period_range(
    start="2007-06",
    end="2018-12",
    freq="M",
).astype("string")

actual_months = pd.Index(
    monthly_group_results["month"]
)

missing_months = expected_months.difference(actual_months)
unexpected_months = actual_months.difference(expected_months)

month_match = (
    actual_months.tolist()
    == expected_months.tolist()
)

print("\n月份是否完全匹配：", month_match)
print("实际月份数量：", len(actual_months))
print("预期月份数量：", len(expected_months))
print("缺失月份：", missing_months.tolist())
print("意外月份：", unexpected_months.tolist())


# 6. 正式验收
assert total_row_count == 2_260_668, "总行数不符合预期"
assert unique_loan_count == 2_260_668, "唯一贷款数不符合预期"
assert duplicate_loan_count == 0, "存在重复贷款 ID"

assert missing_id_count == 0, "存在缺失 id"
assert missing_issue_d_count == 0, "存在缺失 issue_d"
assert missing_issue_month_count == 0, "存在缺失 issue_month"

assert issue_month_mismatch_count == 0, (
    "issue_month 与 issue_d 不一致"
)

assert month_match, "月份顺序或范围不匹配"
assert len(monthly_group_results) == 139, "月份数量不是 139"
assert len(missing_months) == 0, "存在缺失月份"
assert len(unexpected_months) == 0, "存在意外月份"

assert (
    monthly_group_results["total_issued_number"].sum()
    == total_row_count
), "分月后的贷款总数与原数据不一致"

assert (
    monthly_group_results["duplicate_loan_number"].sum()
    == 0
), "月度批次中存在重复贷款"

print("\n✅ Monthly batch audit passed.")

,id,issue_d,issue_month
0,68407277,2015-12-01,2015-12
1,68355089,2015-12-01,2015-12
2,68341763,2015-12-01,2015-12
3,66310712,2015-12-01,2015-12
4,68476807,2015-12-01,2015-12


数据形状： (2260668, 3)

字段类型：
id                      int64
issue_d        datetime64[us]
issue_month            string
dtype: object

基础检查：
总行数： 2260668
唯一贷款数： 2260668
重复贷款数： 0
缺失 id： 0
缺失 issue_d： 0
缺失 issue_month： 0

issue_month 转换不一致数量： 0


,month,total_issued_number,unique_loan_number,earliest_issue_date,latest_issue_date,duplicate_loan_number,cumulative_loan_count
0,2007-06,24,24,2007-06-01,2007-06-01,0,24
1,2007-07,63,63,2007-07-01,2007-07-01,0,87
2,2007-08,74,74,2007-08-01,2007-08-01,0,161
3,2007-09,53,53,2007-09-01,2007-09-01,0,214
4,2007-10,105,105,2007-10-01,2007-10-01,0,319


,month,total_issued_number,unique_loan_number,earliest_issue_date,latest_issue_date,duplicate_loan_number,cumulative_loan_count
134,2018-08,46079,46079,2018-08-01,2018-08-01,0,2093230
135,2018-09,39026,39026,2018-09-01,2018-09-01,0,2132256
136,2018-10,46305,46305,2018-10-01,2018-10-01,0,2178561
137,2018-11,41973,41973,2018-11-01,2018-11-01,0,2220534
138,2018-12,40134,40134,2018-12-01,2018-12-01,0,2260668



月份是否完全匹配： True
实际月份数量： 139
预期月份数量： 139
缺失月份： []
意外月份： []

✅ Monthly batch audit passed.


In [30]:
audit_output_path = Path(
    "outcomes/monthly_batch_audit.csv"
)

# 确保输出文件夹存在
audit_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

# 保存月度审计结果
monthly_group_results.to_csv(
    audit_output_path,
    index=False,
)

print(
    f"✅ Monthly batch audit saved: "
    f"{audit_output_path.resolve()}"
)

✅ Monthly batch audit saved: D:\Project\outcomes\monthly_batch_audit.csv


## 2.2replay

In [31]:
#设定路径
parquet_path = Path(
    "data/processed/accepted_loans_normalized.parquet"
)
audit_path = Path(
    "outcomes/monthly_batch_audit.csv"
)

monthly_audit = pd.read_csv(audit_path)
loans = pd.read_parquet(parquet_path)
expected_counts = monthly_audit.set_index("month")["total_issued_number"]
expected_months = monthly_audit["month"].tolist()
expected_total = monthly_audit["total_issued_number"].sum()

print("数据形状：", loans.shape)
print("字段数量：", len(loans.columns))
print("预期月份数量：", len(expected_months))
print("预期贷款总数：", expected_total)

assert len(loans) == expected_total
assert loans["id"].isna().sum() == 0
assert loans["id"].duplicated().sum() == 0
assert loans["issue_month"].isna().sum() == 0
# 留下源数据 ID，稍后与分区文件进行完整比较
source_ids = (
    loans["id"]
    .sort_values()
    .reset_index(drop=True)
)

print("✅ Partition input validation passed.")


数据形状： (2260668, 166)
字段数量： 166
预期月份数量： 139
预期贷款总数： 2260668
✅ Partition input validation passed.


In [32]:
#创建139个月度分区
batches_root = Path(
    "data/batches"
)

batches_root.mkdir(
    parents=True,
    exist_ok=True,
)

written_file_count = 0
written_row_count = 0

for issue_month,monthly_batch in loans.groupby("issue_month"):
    assert issue_month in expected_months, f"Unexpected month: {issue_month}"
    expected_monthly_count = expected_counts[issue_month]
    assert len(monthly_batch) == expected_monthly_count, f"{issue_month} 的贷款数量与 audit 不一致"
    assert monthly_batch["issue_month"].eq(issue_month).all(),f"{issue_month} 分区混入了其他月份"
    monthly_batch_path = batches_root / f"issue_month={issue_month}"
    monthly_batch_path.mkdir(parents=True, exist_ok=True)
    monthly_batch_file = monthly_batch_path / "loans.parquet"
    monthly_batch.to_parquet(monthly_batch_file, index=False)
    written_file_count += 1
    written_row_count += len(monthly_batch)

assert written_file_count == len(expected_months), "写入的分区文件数量与预期不一致"
assert written_row_count == expected_total, "写入的总行数与预期不一致"

print("\n✅ Monthly partitions created.")
print("分区文件数量：", written_file_count)
print("写出贷款总数：", written_row_count)



✅ Monthly partitions created.
分区文件数量： 139
写出贷款总数： 2260668


In [33]:
#提取partition文件的信息
partition_files = sorted(batches_root.glob("issue_month=*/loans.parquet"))
assert len(partition_files) == len(expected_months), "分区文件数量与预期不一致"

saved_id_parts = []
validation_rows = []
for partition_file in partition_files:
    partition_month = partition_file.parent.name.removeprefix("issue_month=")
    partition_data = pd.read_parquet(partition_file,columns=["id", "issue_month"])
    saved_row_count = len(partition_data)
    saved_unique_count = partition_data["id"].nunique()
    saved_duplicate_count = partition_data["id"].duplicated().sum()
    assert partition_data["issue_month"].eq(
        partition_month
    ).all(), (
        f"{partition_month} 文件中混入其他月份"
    )
    saved_id_parts.append(partition_data["id"].sort_values().reset_index(drop=True))
    validation_rows.append(
        {
            "month": partition_month,
            "saved_row_count": saved_row_count,
            "saved_unique_count": saved_unique_count,
            "saved_duplicate_count": saved_duplicate_count,
        }
    )
    assert saved_row_count == expected_counts[partition_month], f"{partition_month} 分区的行数与 audit 不一致"
    assert saved_unique_count == expected_counts[partition_month], f"{partition_month} 分区的唯一贷款数与 audit 不一致"
    assert saved_duplicate_count == 0, f"{partition_month} 分区存在重复贷款 ID"

partition_validation = (
    pd.DataFrame(validation_rows)
    .sort_values("month")
    .reset_index(drop=True)
)

display(partition_validation.head())
display(partition_validation.tail())

,month,saved_row_count,saved_unique_count,saved_duplicate_count
0,2007-06,24,24,0
1,2007-07,63,63,0
2,2007-08,74,74,0
3,2007-09,53,53,0
4,2007-10,105,105,0


,month,saved_row_count,saved_unique_count,saved_duplicate_count
134,2018-08,46079,46079,0
135,2018-09,39026,39026,0
136,2018-10,46305,46305,0
137,2018-11,41973,41973,0
138,2018-12,40134,40134,0


In [34]:
#将partition文件的信息和audit信息进行对比
saved_ids = pd.concat(saved_id_parts, ignore_index=True).sort_values().reset_index(drop=True)
actual_months = partition_validation["month"].tolist()
assert actual_months == expected_months, (
    "分区月份与 audit 不一致"
)
assert len(saved_ids) == expected_total, (
    "所有分区的贷款总数不正确"
)
assert saved_ids.duplicated().sum() == 0, (
    "不同分区之间存在重复贷款"
)

try:
    pd.testing.assert_series_equal(source_ids, saved_ids, check_names=False, check_dtype=True)
except AssertionError as err:
    print(f"❌ 全量ID校验失败：{err}")
    raise

print("✅ 全量ID校验通过")
print("✅ Physical monthly partition validation passed.")
print("月份数量：", len(partition_validation))
print("贷款总数：", len(saved_ids))
print("重复贷款数：", saved_ids.duplicated().sum())

✅ 全量ID校验通过
✅ Physical monthly partition validation passed.
月份数量： 139
贷款总数： 2260668
重复贷款数： 0
